# S2 · MachineEar tools and structured outputs
Three tools: `analyze_audio` (read), `search_manuals` (read), and `save_report_revision` (write).

This notebook adapts the S2 lab to the real project APIs. Default demonstrations use labelled synthetic audio/retrieval fixtures, Pydantic validation, and real temporary history writes. Live audio and retrieval are separate opt-in cells. No Groq calls, keys, or model downloads occur automatically.

Run from a checkout with `requirements/agent.txt` installed and select its virtual-environment kernel. Existing notebooks and audio models are preserved.


In [ ]:
from pathlib import Path
import sys, json, tempfile
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "machineear" / "audio_bridge.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the MachineEar checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pydantic import ValidationError
from machineear.agent_tools.schemas import tool_schemas
from machineear.agent_tools.executor import ToolExecutor
print("Repository:", ROOT)


## 1. The three schemas
Parameters come from Pydantic. The output schemas document the validated tool results. Access metadata stays outside provider function definitions.

In [ ]:
schemas = tool_schemas()
for entry in schemas:
    print(entry["name"], entry["access"])
    display(Markdown("```json\n" + json.dumps(entry, indent=2) + "\n```"))
assert len(schemas) == 3
provider_tools = [entry["tool"] for entry in schemas]


## 2. Synthetic contract demonstration
These fixtures demonstrate routing and validation only. They are not actual detector outputs or manual-search measurements.

In [ ]:
AUDIO_FIXTURE = {
    "machine_type": "fan", "machine_id": "id_00", "model": "logmel_knn5_mean",
    "anomaly_score": 0.75, "threshold": 0.7, "is_anomaly": True,
    "duration_seconds": 2.0, "n_windows": 1,
    "windows": [{"start_seconds": 0.0, "end_seconds": 2.0, "score": 0.75}]
}
def fixture_predictor(source, *, machine_type, machine_id):
    return dict(AUDIO_FIXTURE, machine_type=machine_type, machine_id=machine_id)

class FixtureRetriever:
    def search(self, query, machine_type, *, language, top_k):
        return {"status": "passages_for_review", "results": [{
            "chunk_id": "synthetic-chunk", "text": "Synthetic passage for contract testing only.",
            "source_relative_path": "synthetic_manual.pdf", "pdf_page_number": 1,
            "citation": "[synthetic_manual.pdf, PDF p. 1]", "semantic_similarity": 0.9,
            "citation_verified": True, "machine_scope": machine_type, "needs_review": False}]}

temp_history = tempfile.TemporaryDirectory(prefix="machineear-s2-")
demo = ToolExecutor(history_dir=temp_history.name, retriever=FixtureRetriever(), predictor=fixture_predictor)
recording_id = demo.register_recording(b"synthetic bytes; not actual WAV data")
audio = demo.execute("analyze_audio", {
    "recording_id": recording_id, "machine_type": "fan", "machine_id": "id_00"})
search = demo.execute("search_manuals", {
    "query": "Which bearing checks are documented?", "machine_type": "fan", "language": "en", "top_k": 5})
print("SYNTHETIC audio:", json.dumps(audio, indent=2))
print("SYNTHETIC retrieval:", json.dumps(search, indent=2))


## 3. Invalid arguments are rejected before execution

In [ ]:
try:
    demo.execute("search_manuals", {
        "query": "bearing grease", "machine_type": "fan", "language": "en", "top_k": "five"})
except ValidationError as exc:
    print("EXPECTED validation rejection:", exc.errors()[0]["msg"])
else:
    raise AssertionError("Invalid arguments were accepted")


## 4. A write needs trusted approval
The first call must fail and write nothing. The next call simulates the trusted human-review UI approving a synthetic report, then uses the real append-only save function in a temporary directory. In production, call `approve_report()` only after actual human review; never expose it as a model tool.

In [ ]:
from machineear.review import new_record, load_history
record = new_record({"audio": audio, "retrieval": {"results": []}}, b"synthetic", "synthetic.wav")
report_id = demo.register_report(record)
reviewed_text = "SYNTHETIC S2 demo. No real maintenance decision or machine approval."
args = {"report_id": report_id, "reviewed_text": reviewed_text, "approval_id": "invented"}
try:
    demo.execute("save_report_revision", args)
except PermissionError as exc:
    print("EXPECTED approval rejection:", exc)
else:
    raise AssertionError("Unapproved save was accepted")
assert not list(Path(temp_history.name).glob("*.json"))

# SIMULATED approval of synthetic data only, from trusted notebook code.
approval_id = demo.approve_report(report_id, reviewed_text, "Synthetic lab reviewer", applicability_checked=True)
saved = demo.execute("save_report_revision", {**args, "approval_id": approval_id})
history, errors = load_history(temp_history.name)
assert not errors and len(history) == 1
assert history[0]["review"]["report"] == reviewed_text
print("REAL temporary save of SYNTHETIC report:", json.dumps(saved, indent=2))
try:
    demo.execute("save_report_revision", {**args, "approval_id": approval_id})
except PermissionError:
    print("EXPECTED: single-use approval cannot be replayed")
else:
    raise AssertionError("Approval replay was accepted")


## 5. Optional live retrieval on MachineEar manuals
Prepare embeddings explicitly in a terminal first: `python -m machineear.rag.local_embeddings --prepare`. Change the switch only when ready. Failures are surfaced rather than replaced by fixture output.

In [ ]:
RUN_LIVE_RETRIEVAL = False
if RUN_LIVE_RETRIEVAL:
    live = ToolExecutor()
    result = live.execute("search_manuals", {
        "query": "What NLGI grease grade is specified for fan bearings?",
        "machine_type": "fan", "language": "en", "top_k": 5})
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("NOT RUN: live retrieval. Enable after preparing local embedding assets.")


## 6. Optional live audio using the unchanged predictor
Supply an original **normal-training WAV** and select the correct family/ID. Do not use held-out test recordings. Do not commit recordings or individual analyses.

In [ ]:
RUN_LIVE_AUDIO = False
WAV_PATH = ""  # Absolute path to your normal-training WAV
MACHINE_TYPE = "fan"
MACHINE_ID = "id_00"
if RUN_LIVE_AUDIO:
    if not WAV_PATH:
        raise ValueError("Set WAV_PATH to an original normal-training WAV first")
    live_audio = ToolExecutor()
    registered_id = live_audio.register_recording(WAV_PATH)
    result = live_audio.execute("analyze_audio", {
        "recording_id": registered_id, "machine_type": MACHINE_TYPE, "machine_id": MACHINE_ID})
    print(json.dumps(result, indent=2))
else:
    print("NOT RUN: live audio. Supply normal-training WAV and matching machine settings.")


## 7. Cleanup and submission
Commit schemas, contracts, wrappers, tests, documentation and this notebook with an `S2:` message. Retain local evidence of live calls for the instructor; synthetic demonstrations alone do not establish real model integration. Remove private outputs and API keys before publication.

In [ ]:
temp_history.cleanup()
print("Temporary synthetic history removed. No project history was changed.")
